In [ ]:
# from google.colab import drive

# drive.mount('/content/drive')

# %cd /content/drive/MyDrive/faster_rcnn
# %cp VOC2007.zip /content
# %cp VOC2012.zip /content
# %cd /content

In [ ]:
from pathlib import Path
import zipfile

data_path = Path("data/")
data_path.mkdir(exist_ok=True)

voc2007_zip_path = Path("VOC2007.zip")
voc2012_zip_path = Path("VOC2012.zip")

if not voc2007_zip_path.exists() or not voc2012_zip_path.exists():
    raise RuntimeError("Dataset not found.")

print("Extracting 2007 dataset ...")

with zipfile.ZipFile(voc2007_zip_path, "r") as zip_ref:
    zip_ref.extractall(data_path)

print(f"Extracting 2012 dataset ...")
with zipfile.ZipFile(voc2012_zip_path, "r") as zip_ref:
    zip_ref.extractall(data_path)

In [1]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
!nvidia-smi

Tue Aug 11 05:24:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from src.backbone import backbone_transform
from src.dataset import get_voc_img_paths_train, get_voc_img_paths_test, create_voc_dataloader

transform = backbone_transform

voc2007_img_paths_train, voc2012_img_paths_train = get_voc_img_paths_train()
voc2007_img_paths_test, voc2012_img_paths_test = get_voc_img_paths_test()

combined_train_img_paths = voc2007_img_paths_train + voc2012_img_paths_train

train_dataloader = create_voc_dataloader(img_paths_list=combined_train_img_paths, transform=transform, batch_size=2, shuffle=True)
test_dataloader = create_voc_dataloader(img_paths_list=voc2007_img_paths_test, transform=transform, batch_size=2, shuffle=True)

In [4]:
from src.backbone import Backbone
from src.rpn import RPN_Head, RPN_Loss
import torch
from pathlib import Path

# Step-1 RPN network: frozen, exists only to generate proposals.
backbone = Backbone().to(device)
rpn_head = RPN_Head(in_channels=1024, mid_channels=512).to(device)

# backbone_checkpoint_path = Path("/content/drive/MyDrive/faster_rcnn/checkpoints/step2_epoch_6.pt")
backbone_checkpoint_path = Path("checkpoints/step2_epoch_6.pt")
backbone_checkpoint = torch.load(backbone_checkpoint_path, map_location=device)

# rpn_checkpoint_path = Path("/content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_10.pt")
rpn_checkpoint_path = Path("checkpoints/step1_epoch_10.pt")
rpn_checkpoint = torch.load(rpn_checkpoint_path, map_location=device)

backbone.load_state_dict(backbone_checkpoint['backbone_state_dict'])
print(f"Loaded Backbone Checkpoint: {backbone_checkpoint_path}")
rpn_head.load_state_dict(rpn_checkpoint['rpn_head_state_dict'])
print(f"Loaded RPN Head Checkpoint: {rpn_checkpoint_path}")

for param in backbone.parameters():
    param.requires_grad = False

# eval() here and never train() again
backbone.eval()


# Fn to Freeze BatchNorm layers in the backbone to avoid updating running stats during training conv_5x layers BN was already frozen during detection_head creation
def freeze_batchnorm(module):
    for m in module.modules():
        if isinstance(m, torch.nn.BatchNorm2d):
            m.eval()
            m.weight.requires_grad = False
            m.bias.requires_grad = False

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 209MB/s]


Loaded Backbone Checkpoint: checkpoints/step2_epoch_6.pt
Loaded RPN Head Checkpoint: checkpoints/step1_epoch_10.pt


In [5]:
loss_criterion = RPN_Loss()

params = list(rpn_head.parameters())

optimizer = torch.optim.SGD(
    params, 
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005)

In [6]:
from pathlib import Path

start_epoch = 0

# checkpoint_dir = Path("/content/drive/MyDrive/faster_rcnn/checkpoints")
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

existing_checkpoints = sorted(checkpoint_dir.glob("step3_epoch_*.pt"),
                              key = lambda p : int(p.stem.split("_epoch_")[1]))

if existing_checkpoints:
    latest_checkpoint = existing_checkpoints[-1]
    print(f"Loading checkpoint: {latest_checkpoint}")

    checkpoint = torch.load(latest_checkpoint, map_location=device)
    
    rpn_head.load_state_dict(checkpoint['rpn_head_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1

else:
    print("No existing checkpoints found. Starting training from scratch.")

for param_group in optimizer.param_groups:
    param_group['lr'] = 0.0001

Loading checkpoint: checkpoints/step3_epoch_8.pt


In [7]:
import torch
import time
from tqdm import tqdm

torch.manual_seed(42)
torch.cuda.manual_seed(42)

num_epochs = 10
loss_lambda = 10

rpn_head.train()

for epoch in range(start_epoch, num_epochs):
    start_time = time.time()

    epoch_cls_loss = 0.0
    epoch_reg_loss = 0.0
    num_batches = 0

    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")

    for batch_imgs, batch_gt_boxes, batch_gt_labels, batch_img_sizes_before_pad in progress_bar:
        batch_imgs = batch_imgs.to(device)
        batch_gt_boxes = [boxes.to(device) for boxes in batch_gt_boxes]         # Not needed as the functions take care, but for safety
        batch_gt_labels = [labels.to(device) for labels in batch_gt_labels]

        with torch.no_grad():
            batch_feature_maps = backbone(batch_imgs)

        batch_cls_logits, batch_rpn_box_deltas, batch_anchors = rpn_head(batch_feature_maps, batch_img_height=batch_imgs.shape[2], batch_img_width=batch_imgs.shape[3])

        cls_loss, reg_loss = loss_criterion(batch_cls_logits,
                                            batch_rpn_box_deltas, 
                                            batch_anchors, 
                                            batch_gt_boxes, 
                                            batch_img_sizes_before_pad)
        
        loss = cls_loss + loss_lambda * reg_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_cls_loss += cls_loss.item()
        epoch_reg_loss += reg_loss.item()
        num_batches += 1

        progress_bar.set_postfix({
            "cls_loss": f"{cls_loss.item():.8f}",
            "reg_loss": f"{reg_loss.item():.8f}",
            "total_loss": f"{loss.item():.8f}"
        })

    avg_cls_loss = epoch_cls_loss / num_batches
    avg_reg_loss = epoch_reg_loss / num_batches

    print(f"Epoch {epoch+1}/{num_epochs} completed in {time.time() - start_time:.2f}s")
    print(f"Average Classification Loss: {avg_cls_loss:.8f}")
    print(f"Average Regression Loss: {avg_reg_loss:.8f}")

    checkpoint_path = checkpoint_dir / f"step3_epoch_{epoch+1}.pt"
    torch.save({
        'epoch': epoch,
        'rpn_head_state_dict': rpn_head.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, checkpoint_path)

    print(f"Checkpoint saved at: {checkpoint_path}")

Epoch 9/10: 100%|██████████| 8276/8276 [14:49<00:00,  9.30batch/s, cls_loss=0.05501538, reg_loss=0.00011085, total_loss=0.05612384]


Epoch 9/10 completed in 889.55s
Average Classification Loss: 0.05510164
Average Regression Loss: 0.00027978
Checkpoint saved at: checkpoints/step3_epoch_9.pt


Epoch 10/10: 100%|██████████| 8276/8276 [14:58<00:00,  9.21batch/s, cls_loss=0.01260306, reg_loss=0.00010593, total_loss=0.01366234]


Epoch 10/10 completed in 898.63s
Average Classification Loss: 0.05505273
Average Regression Loss: 0.00027941
Checkpoint saved at: checkpoints/step3_epoch_10.pt
